In [67]:
import mlflow

mlflow.end_run()

### Step 1 - The Configure MLflow tracking location

In [51]:
!pip install mlflow scikit-learn --quiet

In [52]:
import mlflow
import os

# set the tracking location to our root directory
project_root = os.path.abspath("..")
mlflow.set_tracking_uri(f"file:///{project_root}/mlruns")

print(f"MLflow tracking at: {project_root}/mlruns")

MLflow tracking at: c:\Users\Admin\OneDrive\Documents\GitHub\Muinat_Ridewise_Europe/mlruns


### Step 2 - Loading the Mlflow

In [53]:
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature

# Set tracking directory OUTSIDE notebooks
mlflow.set_tracking_uri(
    "file:///c:/Users/Admin/OneDrive/Documents/GitHub/Muinat_Ridewise_Europe/mlruns"
)

print(f"MLflow version: {mlflow.__version__}")
print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")
print("MLflow loaded successfully!")


MLflow version: 3.6.0
MLflow tracking URI: file:///c:/Users/Admin/OneDrive/Documents/GitHub/Muinat_Ridewise_Europe/mlruns
MLflow loaded successfully!


### Step 3 - Import Standard Libraries

In [69]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    precision_score,
    recall_score,
    mean_squared_error,
    classification_report,
    f1_score
)

### Step 4 - Setting up MLflow Experiment

- Experiment like a project folder
- Every model run goes inside it

In [55]:
# set up a folder for our mlflow experiment
EXPERIMENT_NAME = "RideWise-Churn-Prediction"
mlflow.set_experiment(EXPERIMENT_NAME)

# Make sure that MLFLOW EXPERIMENT_NAME is set to the name of your experiment
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

print(f"Experiment name: {experiment.name}")
print(f"Experiment ID: {experiment.experiment_id}")
print(f"Experiment artifact location: {experiment.artifact_location}")

Experiment name: RideWise-Churn-Prediction
Experiment ID: 969808953743187709
Experiment artifact location: file:///c:/Users/Admin/OneDrive/Documents/GitHub/Muinat_Ridewise_Europe/mlruns/969808953743187709


### Step 5 - Load Processed Balanced DataFrame

In [56]:
X_resampled = pd.read_csv("../data/processed/X_resampled.csv")
y_resampled = pd.read_csv("../data/processed/y_resampled.csv")

print(f"X_resampled shape: {X_resampled.shape}")
print(f"y_resampled shape: {y_resampled.shape}")

# Seperate features and target
X = X_resampled
y = y_resampled

X_resampled shape: (4902, 26)
y_resampled shape: (4902, 1)


In [57]:
resampled_df = pd.DataFrame(X_resampled, columns=X.columns)
resampled_df.head(3)

,age,gender,total_trips,days_since_last_trip,trips_30d,completed_trips_30d,cancelled_trips_30d,total_fare_30d,total_distance_30d,promo_used_trips_30d,...,income_level_medium,signup_channel_Organic,signup_channel_Partnership,signup_channel_Referral,persona_Casual Rider,persona_Commuter,persona_Tourist,Cluster_1,Cluster_2,Cluster_3
0,-0.923077,2,1.157895,0.208333,0.25,0.333333,0.0,0.194435,1.363937e-16,0.0,...,0,0,0,0,0,0,1,0,0,1
1,0.692308,2,0.578947,1.208333,-0.50,-0.666667,0.0,-0.605708,-6.488147e-01,0.0,...,0,0,1,0,0,0,0,0,0,0
2,-0.461538,1,0.315789,-0.458333,0.50,0.333333,1.0,0.539812,5.117574e-01,1.0,...,0,0,0,0,0,0,0,0,0,1


In [58]:
resampled_df["churned"] = y_resampled
resampled_df.head(3)

,age,gender,total_trips,days_since_last_trip,trips_30d,completed_trips_30d,cancelled_trips_30d,total_fare_30d,total_distance_30d,promo_used_trips_30d,...,signup_channel_Organic,signup_channel_Partnership,signup_channel_Referral,persona_Casual Rider,persona_Commuter,persona_Tourist,Cluster_1,Cluster_2,Cluster_3,churned
0,-0.923077,2,1.157895,0.208333,0.25,0.333333,0.0,0.194435,1.363937e-16,0.0,...,0,0,0,0,0,1,0,0,1,0
1,0.692308,2,0.578947,1.208333,-0.50,-0.666667,0.0,-0.605708,-6.488147e-01,0.0,...,0,1,0,0,0,0,0,0,0,0
2,-0.461538,1,0.315789,-0.458333,0.50,0.333333,1.0,0.539812,5.117574e-01,1.0,...,0,0,0,0,0,0,0,0,1,0


##### redefine features and target for balanced data

In [59]:
features = resampled_df.drop("churned", axis=1)
target = resampled_df["churned"]

features.to_csv("../data/processed/features.csv")
target.to_csv("../data/processed/target.csv")

print(f"Features shape: {features.shape}")
print(f"Target shape: {target.shape}")

Features shape: (4902, 26)
Target shape: (4902,)


### Step 6: Data Splitting

In [60]:
# Split the dataset (70% training, 30% testing)
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.3, 
                                                    random_state=42, stratify=y_resampled)

print("Training set size:", len(X_train))
print("Testing set size:", len(X_test))

print(f"y_train value counts distribution: {y_train.value_counts(normalize=True)}")
print(f"y_test value counts distribution: {y_test.value_counts(normalize=True)}")

Training set size: 3431
Testing set size: 1471
y_train value counts distribution: churned
1          0.500146
0          0.499854
Name: proportion, dtype: float64
y_test value counts distribution: churned
0          0.50034
1          0.49966
Name: proportion, dtype: float64


### Step 7 - Helper Function- Metric calculation (accuracy etc)

In [61]:
def calculate_metrics(y_true, y_pred_proba, top_percentile=15):
    y_true = y_true.values.ravel()  # flatten the target array if it's a DataFrame
    y_pred_proba = y_pred_proba.ravel()  # flatten the predicted probabilities if needed

    # step 7a - calculate the AUC_ROC score
    # if AUC_ROC is 0.61 (the model is right 61 times out of 100 when its comparing
    # churners vs non-churners)
    auc = roc_auc_score(y_true, y_pred_proba)

    # step 7b - calculate the precious Top 15% or riskest customers
    # Find the minimum score/probaility to be in the top 15% of risk
    threshold = np.percentile(y_pred_proba, 100 - top_percentile)


    # filter the 15% from the general probabilyt data
    # e.g probab 0.72 >= 0.70 (threshold) --> 1 (target)
    # e.g probab 0.65 <= 0.70 (threshold) --> 0 (ignore)
    pred_top = (y_pred_proba >= threshold).astype(int)


    # what happens behind the scene 
    # pred_prob = [0.72, 0.65, 0.80, 0.55, 0.90]
    # threshold = 0.70
    # pred_top = [1,0,1,1,1,0]
    # pred_top.sum() --- 4 (4 customers are in the top 15% of risk)
    # if pred_top.sum() greater than 0 (the model can somehow select 0 customers)
    # division by zero will crash the program 

     # safety check
    if pred_top.sum() > 0:
        precision_at_k = (pred_top[y_true == 1].sum())/ pred_top.sum()
        recall_at_k = (pred_top[y_true == 1].sum())/ (y_true == 1).sum()
    
    else:
        precision_at_k = 0.0
        recall_at_k = 0.0
    
    y_pred = (y_pred_proba >= 0.5).astype(int)

    return {
        "auc_roc": auc,
        f"precision_at_{top_percentile}": precision_at_k,
        f"recall_at_{top_percentile}": recall_at_k,

        "precision": precision_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred),
        "f1_score" : f1_score(y_true, y_pred)
    }

### Step 8 - Experiment 1 - Baseline Logistic regression

In [62]:
with mlflow.start_run(run_name="Logistic Regression"):
    mlflow.log_param("model_type", "Logistic Regression")
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_param("max_iter", 1000)
    mlflow.log_param("random_state", 42)
    mlflow.log_param("n_features", X_train.shape[1])

    # step 8a - train the model

    model = LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42,
    )

    model.fit(X_train,y_train)

    # step 8b - make predictions
    y_pred_proba = model.predict_proba(X_test)[:,1]

    # step 8c - calculate metrics
    metrics = calculate_metrics(y_test, y_pred_proba)

    # step 8d - log metrics to mlflow
    for metric_name, metric_value in metrics.items():
        mlflow.log_metric(metric_name, metric_value)

    # step 8e - log the signature of the model (input and output schema)
    signature = infer_signature(X_train, y_pred_proba)
    mlflow.sklearn.log_model(
        model,
        "model", 
        signature=signature
        )

 # step 8f - log the X_resampled and y_resampled names   
mlflow.log_dict({"features": features.columns.tolist()}, "features.json")

# print out results
print("Logistic Regression model to MLFLOW")



c:\Users\Admin\anaconda3\Lib\site-packages\sklearn\utils\validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
c:\Users\Admin\anaconda3\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more detai

Logistic Regression model to MLFLOW


### Step 8 - Experiment 2 - Random Forest Classifier

In [65]:
with mlflow.start_run(run_name="Random Forest Classifier"):
    mlflow.log_param("model_type", "Random Forest Classifier")
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_param("n_estimators", 300)
    mlflow.log_param("random_state", 42)
    mlflow.log_param("max_depth", 10)
    mlflow.log_param("n_jobs", -1)
    mlflow.log_param("n_features", X_train.shape[1])

    # step 8a - train the model

    model = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

    model.fit(X_train,y_train)

    # step 8b - make predictions
    y_pred_proba = model.predict_proba(X_test)[:,1]

    # step 8c - calculate metrics
    metrics = calculate_metrics(y_test, y_pred_proba)

    # step 8d - log metrics to mlflow
    for metric_name, metric_value in metrics.items():
        mlflow.log_metric(metric_name, metric_value)

    # step 8e - log the signature of the model (input and output schema)
    signature = infer_signature(X_train, y_pred_proba)
    mlflow.sklearn.log_model(
        model,
        "model", 
        signature=signature
        )

 # step 8f - log the X_resampled and y_resampled names   
mlflow.log_dict({"features": features.columns.tolist()}, "features.json")

# print out results
print("Random Forest Classifier model to MLFLOW")


c:\Users\Admin\anaconda3\Lib\site-packages\sklearn\base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
c:\Users\Admin\anaconda3\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more det

Random Forest Classifier model to MLFLOW


### Step 8 - Experiment 3 - Xgboost

In [70]:
with mlflow.start_run(run_name="XGBClassifier"):
    mlflow.log_param("model_type", "XGBClassifier")
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_param("n_estimators", 300)
    mlflow.log_param("random_state", 42)
    mlflow.log_param("objective", "binary:logistic")
    mlflow.log_param("n_jobs", -1)
    mlflow.log_param("eval_metric", "auc")
    mlflow.log_param("use_label_encoder", False)
    mlflow.log_param("n_features", X_train.shape[1])

    # step 8a - train the model

    model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    use_label_encoder=False
)

    model.fit(X_train,y_train)

    # step 8b - make predictions
    y_pred_proba = model.predict_proba(X_test)[:,1]

    # step 8c - calculate metrics
    metrics = calculate_metrics(y_test, y_pred_proba)

    # step 8d - log metrics to mlflow
    for metric_name, metric_value in metrics.items():
        mlflow.log_metric(metric_name, metric_value)

    # step 8e - log the signature of the model (input and output schema)
    signature = infer_signature(X_train, y_pred_proba)
    mlflow.sklearn.log_model(
        model,
        "model", 
        signature=signature
        )

 # step 8f - log the X_resampled and y_resampled names   
mlflow.log_dict({"features": features.columns.tolist()}, "features.json")

# print out results
print("XGBClassifier model to MLFLOW")


c:\Users\Admin\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [23:29:11] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\Admin\anaconda3\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  war

XGBClassifier model to MLFLOW
